## <font color = '#6495ED'>  ______________________ **Text Mining - Stock Sentiment** _____________________
#### _______________________________________________Academic Year: 2024/2025 _____________________________________________

<div style="text-align: center;">
    <strong>Group 31</strong>
    <table style="margin: 0 auto; border-collapse: collapse; border: 1px solid black;">
        <tr>
            <th style="border: 1px solid white; padding: 8px;">Name</th>
            <th style="border: 1px solid white; padding: 8px;">Student ID</th>
        </tr>
        <tr>
            <td style="border: 1px solid white; padding: 8px;">David Cascão</td>
            <td style="border: 1px solid white; padding: 8px;">20240851</td>
        </tr>
        <tr>
            <td style="border: 1px solid white; padding: 8px;">Elcano Gaspar</td>
            <td style="border: 1px solid white; padding: 8px;">20241021</td>
        </tr>
        <tr>
            <td style="border: 1px solid white; padding: 8px;">Jorge Cordeiro</td>
            <td style="border: 1px solid white; padding: 8px;">20240594</td>
        </tr>
        <tr>
            <td style="border: 1px solid white; padding: 8px;">Rui Reis</td>
            <td style="border: 1px solid white; padding: 8px;">20240854</td>
        </tr>
    </table>
</div>

### <font color = '#6495ED'>  **1. Libraries, Functions, Configuration and Datasets** </font>

#### <font color = '#6495ED'> **1.1. Import the Needed Libraries** </font>

In [13]:
#Data management
import numpy as np
import pandas as pd
import csv

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from scipy.stats import pointbiserialr

from collections import Counter


from sklearn.metrics import classification_report

#Preprocessing
import string
from tqdm import tqdm
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

#Visualization
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns
from textblob import TextBlob
import plotly.express as px

import time
from tqdm import tqdm

#### <font color = '#6495ED'> **1.2. Setup and Configuration** </font>

In [14]:
# Download required NLTK resources (run once)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
pd.set_option('display.max_colwidth', None)
# Set seaborn style and matplotlib defaults
sns.set(style="whitegrid", color_codes= "#152645", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.dpi'] = 100

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


#### <font color = '#6495ED'>**1.3. Load the Dataset** </font>

In [15]:
train = pd.read_csv("/content/train.csv")


### <font color = '#6495ED'> **3. Preprocessing** </font>

#### <font color = '#6495ED'> **3.1. Corpus split** </font>

In [16]:
val, train = train_test_split(train, test_size=0.2,random_state=42,stratify=train["label"])

#### <font color = '#6495ED'> **3.2. Data Cleaning for Generative Models** </font>

In [17]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 10.1 MB/s eta 0:00:00


In [18]:
import re
import emoji
from tqdm import tqdm

def clean_for_generative_models(text_list):

    cleaned = []

    for text in tqdm(text_list, desc="Cleaning texts"):
        # Remove URLs
        text = re.sub(r"http\S+|www\S+|https\S+", "", text)
        # Remove mentions
        text = re.sub(r"@\w+", "", text)
        # Remove hashtags (keep the word)
        text = re.sub(r"#(\w+)", r"\1", text)
        # Remove emojis
        text = emoji.replace_emoji(text, replace='')
        # Remove extra whitespace and unwanted symbols
        text = re.sub(r"[“”‘’•]", "", text)
        text = re.sub(r"[^\w\s.,!?$%-]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        cleaned.append(text)

    return cleaned

train["clean_text"] = clean_for_generative_models(train["text"].tolist())

Cleaning texts: 100%|██████████| 1909/1909 [00:00<00:00, 11546.86it/s]


In [19]:
train.sample(5,random_state=25)

,text,label,clean_text
7810,$AUPH - Aurinia's Successful Study In Lupus Nephritis Changes The Scope Of Treatment Options For Patients. Follow t… https://t.co/DbMxN5iCwi,1,$AUPH - Aurinias Successful Study In Lupus Nephritis Changes The Scope Of Treatment Options For Patients. Follow t
8393,Are Kapsch TrafficCom AG’s Returns On Capital Worth Investigating?,2,Are Kapsch TrafficCom AGs Returns On Capital Worth Investigating?
5325,Thriving in a trade war: Japan's Murata finds a way https://t.co/Bw89pdztqe,2,Thriving in a trade war Japans Murata finds a way
1071,$ALXN said that it will maintain an active dialogue with shareholders and welcomes constructive impact” https://t.co/nPMG69yiLb,2,$ALXN said that it will maintain an active dialogue with shareholders and welcomes constructive impact
7676,$ING - ING May Be Through The Worst Point Of The Cycle. Read more: https://t.co/wTGUTspx6E #stockmarket #economy #investing,0,$ING - ING May Be Through The Worst Point Of The Cycle. Read more stockmarket economy investing


#### <font color = '#6495ED'> **3.3. Generative Models for Classification** </font>

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

# Define model name and cache directory
model_name = "microsoft/phi-3-mini-4k-instruct"
cache_dir = "/content/model_cache"

# Load tokenizer and model using accelerate (loads to GPU automatically)
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    torch_dtype=torch.float16,
    device_map="auto"  # Automatically uses GPU if available
)

# Create text generation pipeline
text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("✅ Phi-3 Mini model loaded and running on GPU (if available)")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Phi-3 Mini model loaded and running on GPU (if available)


In [20]:
from sklearn.metrics import classification_report
def evaluate_performance(y_true, y_pred):
    print(classification_report(y_true, y_pred, target_names=["Bearish", "Bullish", "Neutral"]))

In [21]:
def zero_shot_prompt(text):
    return f"""You are a financial sentiment classifier.

Your task is to classify the sentiment of the following financial news text into one of the three categories:

- 0: Bearish → Negative sentiment or expectation of decline.
- 1: Bullish → Positive sentiment or expectation of growth.
- 2: Neutral → No clear sentiment or purely factual.

Text:
{text}

Respond only with the number 0, 1, or 2. No explanation needed.
"""


In [22]:
def classify_with_model(prompt):
    response = text_generator(
        prompt,
        max_new_tokens=10,
        do_sample=False
    )[0]["generated_text"]
    matches = re.findall(r"\b[012]\b", response)
    return int(matches[-1])

In [23]:
expirement=train.sample(5)
expirement

,text,label,clean_text
3360,"$AMKR - Amkor Technology EPS beats by $0.18, beats on revenue https://t.co/A5MDjQ8u4o",1,"$AMKR - Amkor Technology EPS beats by $0.18, beats on revenue"
1969,"Troy Income & Growth Trust : U.S. Dairy Industry Commends Breakthrough on USMCA, Urges Swift Passage of Deal… https://t.co/78LUyoVp0S",2,"Troy Income Growth Trust U.S. Dairy Industry Commends Breakthrough on USMCA, Urges Swift Passage of Deal"
7536,Barclays head of corporate broking Kunal Gandhi has left - sources https://t.co/ny94ERQRy6 https://t.co/e8UngbkwnU,2,Barclays head of corporate broking Kunal Gandhi has left - sources
7285,Merkel Calls for Reversal of CDU's 'Unforgivable' Far-Right Deal,2,Merkel Calls for Reversal of CDUs Unforgivable Far-Right Deal
8069,RECAP 11/22 +Pos Comments:\n$APTV + Wolfe\n$ALV + Wolfe\n$VC + Wolfe\n$HMLP + DNB Markets\n$HEPA + Brookline,1,RECAP 1122 Pos Comments $APTV Wolfe $ALV Wolfe $VC Wolfe $HMLP DNB Markets $HEPA Brookline


In [24]:
for text in expirement["clean_text"]:
    prompt = zero_shot_prompt(text)
    prediction = classify_with_model(prompt)
    print(f"Text: {text}, Prediction: {prediction}\n")

Text: $AMKR - Amkor Technology EPS beats by $0.18, beats on revenue, Prediction: 1

Text: Troy Income Growth Trust U.S. Dairy Industry Commends Breakthrough on USMCA, Urges Swift Passage of Deal, Prediction: 2

Text: Barclays head of corporate broking Kunal Gandhi has left - sources, Prediction: 2

Text: Merkel Calls for Reversal of CDUs Unforgivable Far-Right Deal, Prediction: 2

Text: RECAP 1122 Pos Comments $APTV Wolfe $ALV Wolfe $VC Wolfe $HMLP DNB Markets $HEPA Brookline, Prediction: 1



In [ ]:
from tqdm import tqdm

train_preds_zero = []
for text in tqdm(train["clean_text"], desc="Classifying training data"):
    prompt = zero_shot_prompt(text)
    prediction = classify_with_model(prompt)
    train_preds_zero.append(prediction)


Classifying training data:   2%|▏         | 38/1909 [59:50<49:30:50, 95.27s/it]

In [ ]:
evaluate_performance(train["label"].tolist(), train_preds_zero)

##### <font color = '#6495ED'> **3.3.1. Few-Shot Classification** </font>

In [ ]:
few_shot_examples = [
    ("Deutsche Bank sues over $1.6 billion in claims against Bernard Madoff’s bankrupt investment advisory business", 0),
    ("Axcelis Technologies stock price target raised to $32 from $24 at Benchmark", 1),
    ("Web inventor has an ambitious plan to take back the net", 2),
]

def few_shot_prompt(text, examples=few_shot_examples):
    prompt = """You are a financial sentiment classifier.

Categories:
- 0 = Bearish (negative)
- 1 = Bullish (positive)
- 2 = Neutral (no sentiment)

Classify each text below:

"""
    for ex_text, ex_label in examples:
        prompt += f'Text: "{ex_text}"\nSentiment: {ex_label}\n\n'

    prompt += f'Text: "{text}"\nSentiment:'
    return prompt


In [ ]:
for text in expirement["clean_text"]:
    prompt = few_shot_prompt(text)
    prediction = classify_with_model(prompt)
    print(f"Text: {text}, Prediction: {prediction}\n")

In [ ]:
from tqdm import tqdm

train_preds_few = []
for text in tqdm(train["clean_text"], desc="Classifying training data"):
    prompt = few_shot_prompt(text)
    prediction = classify_with_model(prompt)
    train_preds_few.append(prediction)


In [ ]:
evaluate_performance(train["label"].tolist(), train_preds_few)

In [ ]:
from sklearn.metrics import confusion_matrix
def plot_confusion_matrix(y_true, y_pred, labels=["Bearish", "Bullish", "Neutral"], title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    # Plot
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_normalized, annot=cm, fmt='d', cmap="Blues", xticklabels=labels, yticklabels=labels, linewidths=0.5, cbar=False)
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_confusion_matrix(train["label"].tolist(),train_preds_zero)

In [ ]:
plot_confusion_matrix(train["label"].tolist(),train_preds_few)